In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from experiments import evaluation as ev

In [ ]:
df = ev.build_results_short("results")

In [ ]:
df.to_csv("results_1.csv", index=False)

In [ ]:
long_df = ev.wide_to_long_runs(df)
ev.plot_runs_vs_gold_big(long_df, title="Gold vs Runs – Overview")

In [ ]:
long_err = ev.errors_wide_to_long(df)
summary = ev.summarize_errors(long_err)
summary

In [ ]:
mean_conf_error_matrix = (
    summary[summary["type"] == "conf_error"]
    .pivot(index="run", columns="domain", values="mean")
    .sort_index()
)

mean_conf_error_matrix

In [ ]:
mean_error_matrix = (
    summary[summary["type"] == "error"]
    .pivot(index="run", columns="domain", values="mean")
    .sort_index()
)

mean_error_matrix

In [ ]:
total_per_run = (
    long_err
    .groupby(["type", "run"], as_index=False)
    .agg(
        N=("value", "count"),
        mean=("value", "mean"),
        sum=("value", "sum"),
        max=("value", "max"),
    )
    .sort_values(["type", "run"])
)
total_per_run

In [ ]:
df_totals = ev.total_error_per_run(df)
df_totals

In [ ]:
df_cat = ev.error_summary_per_category(df)
df_cat

In [ ]:
(df == 9).sum()


In [ ]:
import tiktoken

OUTPUT_CONTRACT_DEFAULT = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "A": {"severity": <int>},
  "B": {"severity": <int>},
  "C": {"severity": <int>},
  "D": {"severity": <int>},
  "E": {"severity": <int>}
}
""".strip()

prompt = ("You are an expert in analysing emergency calls. \n"
              "You should evaluate the call transcript according the ABCDE schema. \n"
              "ABCDE Schema: Airway, Breathing, Circulation, Disability, Exposure.\n"
              "Give every category a severity score from 0 to 3.\n"
              "**0** – No information available; no conclusions possible\n"  
            "**1** – No or only minor impairment suspected\n"  
            "**2** – Significant impairment present; no acute life threat suspected (ambulance indicated)\n"  
            "**3** – Severe impairment; vital threat possible or likely; immediate intervention required (urgent ambulance, possibly physician response unit)")

text = prompt + "\n\n" + OUTPUT_CONTRACT_DEFAULT


## Token zählen: 


encoding = tiktoken.get_encoding("o200k_base")

tokens = encoding.encode(text)

print("GPT 5.2:" , len(tokens))

In [ ]:
import anthropic
import pandas as pd
import requests
from dotenv import load_dotenv
from anthropic import Anthropic
import os

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

response = client.messages.count_tokens(
    model="claude-opus-4-6",
    messages=[{"role": "user", "content": text}],
)

print(response.json())